In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")  # all-MiniLM-L6-v2 if you want faster but noisier results
model

/workspaces/applsoftcomp-sprint-m04/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 484.97it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [2]:
import pandas as pd
uni_df = pd.read_csv("../data/universities.csv")



In [3]:
uni_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    156 non-null    str  
 1   type    156 non-null    str  
 2   region  156 non-null    str  
dtypes: str(3)
memory usage: 3.8 KB


In [4]:

print(f"{len(uni_df)} unis across {uni_df['region'].nunique()} regions.")
uni_df.head()


156 unis across 4 regions.


,name,type,region
0,Brown University,Ivy,Northeast
1,Columbia University in the City of New York,Ivy,Northeast
2,Cornell University,Ivy,Northeast
3,Dartmouth College,Ivy,Northeast
4,Harvard University,Ivy,Northeast


In [10]:

print(f"{len(uni_df)} unis across {uni_df['type'].nunique()} types.")
uni_df.head(50)


156 unis across 14 types.


,name,type,region
0,Brown University,Ivy,Northeast
1,Columbia University in the City of New York,Ivy,Northeast
2,Cornell University,Ivy,Northeast
3,Dartmouth College,Ivy,Northeast
4,Harvard University,Ivy,Northeast
5,Princeton University,Ivy,Northeast
6,University of Pennsylvania,Ivy,Northeast
7,Yale University,Ivy,Northeast
8,George Washington University,Elite Private,South
9,Johns Hopkins University,Elite Private,South


In [24]:
uni_df['type'].unique()


<StringArray>
[              'Ivy',     'Elite Private',              'Tech',
   'Public Flagship',   'Public Regional',      'Liberal Arts',
              'HBCU',         'Religious',            'Womens',
   'Service Academy',        'For-Profit', 'Community College',
            'Tribal',  'Specialized Arts']
Length: 14, dtype: str

In [21]:
import numpy as np

def make_axis(positive_words, negative_words, embedding_model):
    """Return a unit-length semantic axis from two word sets."""

    # get the embeddings for each pole
    pos_emb = embedding_model.encode(positive_words, normalize_embeddings=True)
    neg_emb = embedding_model.encode(negative_words, normalize_embeddings=True)

    # Compute the pole centroids
    # axis = 0 means "average across the rows, keep the columns (dims) intact"
    # since pos_emb is shape (num_pos_words, embedding_dim), the mean is shape (embedding_dim,)
    pole_pos = pos_emb.mean(axis=0)  # (embedding_dim,)
    pole_neg = neg_emb.mean(axis=0)  # (embedding_dim,)

    # The axis is the difference between the two centroids, normalized to unit length.
    v = pole_pos - pole_neg

    v = v / (np.linalg.norm(v) + 1e-10)  # add small epsilon to prevent division by zero

    return v / (np.linalg.norm(v) + 1e-10)

In [22]:
def score_words(words, axis, embedding_model):
    """Project each word onto the axis. Returns one score per word."""

    emb = embedding_model.encode(list(words), normalize_embeddings=True)

    # Projection to the axis is just a dot product (since the axis is unit-length).
    # @ is matrix multiplication in NumPy. Since `emb` is shape (num_words, embedding_dim) and `axis` is shape (embedding_dim,), the result is shape (num_words,), which is exactly what we want: one score per word.
    proj = emb @ axis

    return proj

In [25]:
# semantic axes
axis1_pos = [
    "elite",
    "selective",
    "prestigious",
]

axis1_neg = [
    "accessible",
    "open",
    "local",
]

In [26]:
axis_prestige = make_axis(axis1_pos, axis1_neg, model)

In [28]:
axis2_pos = [
    "urban",
    "big city",
    "downtown",
]
axis2_neg = [
    "small town",
    "suburban",
    "rural",
]


In [29]:
axis_focus = make_axis(axis2_pos, axis2_neg, model)